In [ ]:
torch.cuda.get_device_name(0)

In [1]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import warnings

from pytorch_lightning.callbacks import EarlyStopping
# Import our custom data access client
# Restart imports to pick up new methods
import importlib
import src.aqf.data_access
import src.aqf.tickers
import src.aqf.dcc_lightning
importlib.reload(src.aqf.data_access)
importlib.reload(src.aqf.tickers)
importlib.reload(src.aqf.dcc_lightning)
from src.aqf.data_access import FirstRateDataClient
from src.aqf.tickers_full import NASDAQ_TICKERS
from src.aqf.dcc_lightning import StockVolDataModule, DilatedCausalCNN, LossHistory

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
warnings.filterwarnings('ignore')

# Configure matplotlib
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

# Configure plotly
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
# Initialize the FirstRateData client
client = FirstRateDataClient(profile_name="firstratedata")

# Get overview of available data
print("\nData Availability Overview:")
dates_info = client.get_available_dates()
for key, value in dates_info.items():
    print(f"   {key}: {value}")

In [ ]:
start_date = '2021-04-01'
end_date = '2021-09-30' 

print(f"\nLoading {len(NASDAQ_TICKERS)} tickers from {start_date} to {end_date} ...")

try:
    multi_ticker_data = client.load_multi_ticker_data(
        tickers=NASDAQ_TICKERS,
        start_date=start_date,
        end_date=end_date,
        frequency='1T',  
        add_features=True,
        fill_method='drop'
    )
    
    print(f"Successfully loaded data!")
    
    # Display sample data
    print(f"\nSample data (first 5 rows):")
    print(multi_ticker_data.head())
    
except Exception as e:
    print(f"Error loading multi-ticker data: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
if 'multi_ticker_data' in locals() and not multi_ticker_data.empty:
    try:
        # Create aligned dataset (tickers as columns)
        aligned_data = client.create_aligned_dataset(
            multi_ticker_data, 
            value_column='close',  # Use close prices
            fill_method='forward'
        )
        
        print(f"Successfully created aligned dataset!")
        print(f"   Missing values per ticker:")
        for col in aligned_data.columns:
            missing = aligned_data[col].isna().sum()
            print(f"      {col}: {missing} ({missing/len(aligned_data)*100:.1f}%)")
        
        # Display sample aligned data
        print(f"\nSample aligned data (first 5 rows):")
        print(aligned_data.head())
        
        # Plot aligned data
        fig, ax = plt.subplots(figsize=(15, 8))
        for ticker in aligned_data.columns:
            ax.plot(aligned_data.index, aligned_data[ticker], label=ticker, alpha=0.8)
        
        ax.set_title('Aligned Stock Prices', fontsize=16)
        ax.set_xlabel('Time', fontsize=12)
        ax.set_ylabel('Price ($)', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error creating aligned dataset: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No multi-ticker data available to create aligned dataset.")

In [ ]:
aligned_data = pd.read_pickle('1min_data_part1.pkl')


In [ ]:
d = pd.read_pickle('1min_data_part5.pkl')
aligned_data = pd.concat([aligned_data, d], axis=0)
aligned_data

In [ ]:
df = aligned_data_corr
num_days = 1
mode = "univariate"
ticker_split = False
batch_size = 512
# val_start_date = '2020-10-01'
# test_start_date = '2021-01-01'
val_start_date = '2020-10-01'
test_start_date = '2021-01-01'
residual_channels=64
skip_channels=128
end_channels=64
kernel_size=3
num_blocks=2
num_layers=6
dropout=0.0
lr= 1e-3
patience=50
max_epochs=1000

In [ ]:
datamodule = StockVolDataModule(df=df, val_start_date=val_start_date, test_start_date=test_start_date,
                                 num_days=num_days, mode=mode, ticker_split=ticker_split, batch_size=batch_size)

In [ ]:
num_stocks = df.shape[1]
model = DilatedCausalCNN(
    in_channels=(num_stocks if mode=="multivariate" else 1),   # features = number of stocks
    out_channels=(num_stocks if mode=="multivariate" else 1),  # predict volatility for each stock
    residual_channels=residual_channels,
    skip_channels=skip_channels,
    end_channels=end_channels,
    kernel_size=kernel_size,
    num_blocks=num_blocks,
    num_layers=num_layers,
    dropout=dropout,
    lr=lr,
)

early_stop = EarlyStopping(monitor="val_loss", patience=patience, mode="min")

history = LossHistory()
trainer = pl.Trainer(
    max_epochs=max_epochs,
    enable_checkpointing=False,
    callbacks=[early_stop, history],
    log_every_n_steps=1,
)

In [ ]:
aligned_data_corr = (
    aligned_data
    .groupby(df.index.date)            # per day
    .apply(lambda x: x.reindex(
        pd.date_range(
            x.index.min().normalize() + pd.Timedelta("9h30m"),
            x.index.min().normalize() + pd.Timedelta("16h"),
            freq="1min"
        )
    ))
    .droplevel(0)
    .ffill()
)


In [ ]:
aligned_data_corr.index.diff(1).value_counts()

In [ ]:
trainer.fit(model, datamodule=datamodule)

# Plot
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(history.train_losses, label="Training Loss")
plt.plot(history.val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale('log')
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
x, y = datamodule.train_dataset[0]
print(x.mean().item(), x.std().item())

In [ ]:
datamodule.full_dataset._X_mean, datamodule.full_dataset._X_std

In [ ]:
trainer.test(model, datamodule=datamodule)

In [ ]:
import numpy as np

def scatter_pred_vs_real(y_true, y_pred, tickers=None):
    """
    y_true, y_pred: shape (N, A)
    tickers: optional list of ticker names of length A
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    N, A = y_true.shape
    if tickers is None:
        tickers = [f"Asset {i}" for i in range(A)]

    colors = plt.cm.tab20(np.linspace(0, 1, A))

    plt.figure(figsize=(7, 6))
    for i in range(A):
        plt.scatter(y_true[:, i], y_pred[:, i], color=colors[i], label=tickers[i], alpha=0.7)

    # reference line
    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], "k--", lw=2)

    plt.xlabel("Realized Volatility")
    plt.ylabel("Predicted Volatility")
    plt.legend(ncol=2, fontsize=7)
    plt.title("Predicted vs Realized Volatility")
    plt.grid(True)
    plt.tight_layout()
    plt.yscale('log')
    plt.xscale('log')
    plt.show()


In [ ]:
y_pred = model.test_preds
y_true = model.test_targets
scatter_pred_vs_real(y_true, y_pred, tickers=df.columns.tolist())

In [ ]:
min(y_true.flatten())

In [ ]:
datamodule.train_dataset.dataset.samples[1]

In [2]:
df = pd.read_pickle('full_data_1min.pkl')

In [ ]:
df